# 실습 문제 - 103. RAG Agent

`# TODO` 표시된 부분을 채우며 위에서부터 셀 단위로 실행하세요. (모범답안: `practice_answer_103_RAG_Agent.ipynb`)

### 문제 요약

**블로그 QA 에이전트 (인덱싱 + 검색 도구 + RAG 에이전트)**

1. **인덱싱**: `WebBaseLoader`로 블로그 로드 → `RecursiveCharacterTextSplitter`로 분할 → 새 `InMemoryVectorStore`에 저장
2. **검색 도구**: `retrieve_blog_context(query)` - k=3 검색, `content_and_artifact` 형식으로 반환
3. **RAG 에이전트**: 검색된 정보에 근거해서만 답변하도록 system prompt 지정 (환각 방지)
4. **테스트**: 블로그 관련 질문 + 블로그에 없는 질문으로 근거 기반 답변 여부 확인

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 모델 및 임베딩 초기화
model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 인덱싱: 웹 문서 로드 → 분할 → 벡터 스토어 저장

본문의 나이키 벡터 스토어와 섞이지 않도록 **별도의 새 벡터 스토어**를 만듭니다.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) 웹 문서 로드
loader = WebBaseLoader("https://botpress.com/ko/blog/llm-agents")
docs = loader.load()
print(f"로드된 문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content)} 문자")

# 2) 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    # TODO: 청크 크기 1000자, 겹침 200자로 지정하세요.
    chunk_size=...,
    chunk_overlap=...,
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"분할된 청크 수: {len(all_splits)}")

# 3) 새 벡터 스토어에 저장 (본문의 나이키 벡터 스토어와 분리)
blog_vector_store = InMemoryVectorStore(embeddings)
# TODO: 분할된 청크(all_splits)를 벡터 스토어에 추가하세요.
document_ids = ...
print(f"저장된 문서 ID 수: {len(document_ids)}")

## 2. 검색 도구 정의

`response_format="content_and_artifact"` 를 사용하여 LLM에게 전달할 문자열(content)과
원본 문서 객체(artifact)를 함께 반환합니다.

In [ ]:
from langchain.tools import tool

# TODO: LLM용 문자열과 원본 문서를 함께 반환하는 response_format 을 지정하세요.
@tool(response_format=...)
def retrieve_blog_context(query: str):
    """블로그 게시글에서 질문과 관련된 정보를 검색합니다.
    
    Args:
        query: 검색할 질문 또는 키워드
    """
    # 벡터 스토어에서 유사한 문서 검색 (k=3)
    # TODO: 벡터 스토어에서 query 와 유사한 문서 3개를 검색하세요.
    retrieved_docs = ...
    
    # 검색된 문서를 출처 포함 문자열로 직렬화
    serialized = "\n\n".join(
        f"출처: {doc.metadata.get('source', 'N/A')}\n내용: {doc.page_content}"
        for doc in retrieved_docs
    )
    
    # TODO: (LLM에게 줄 문자열, 원본 문서 리스트) 순서로 반환하세요.
    return ...

# 도구 단독 테스트
test_result = retrieve_blog_context.invoke("LLM 에이전트의 구성 요소")
print(test_result[:300] + "...")

## 3. RAG 에이전트 생성

검색된 정보에 근거해서만 답변하도록 system prompt에 지침을 명시합니다.
이렇게 하면 지식 베이스에 없는 질문에 대해 그럴듯한 답을 지어내는 환각(hallucination)을 줄일 수 있습니다.

In [ ]:
from langchain.agents import create_agent

system_prompt = (
    "당신은 블로그 게시글에서 관련 문맥(context)을 검색하는 도구에 접근할 수 있습니다. "
    "사용자의 질문에 답하기 전에 반드시 retrieve_blog_context 도구로 관련 정보를 검색하세요. "
    # TODO: 검색된 정보에 근거해서만 답하고, 검색 결과에 없는 내용은 모른다고 답하라는 지침 문장을 추가하세요.
)

blog_agent = create_agent(
    model,
    # TODO: 검색 도구를 등록하세요.
    tools=...,
    system_prompt=system_prompt
)

## 4. 테스트

In [ ]:
queries = [
    "LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?",
    # TODO: 블로그에 없는 내용을 묻는 질문을 하나 추가해 '모른다'고 답하는지 확인하세요.
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"질문: {query}")
    print("="*80)
    
    # 스트리밍 방식으로 실행하여 도구 호출 여부를 단계별로 확인
    for event in blog_agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values",
    ):
        event["messages"][-1].pretty_print()